In [ ]:
import pandas as pd

books = pd.read_csv("books_cleaned.csv")

In [ ]:
# Explore the distribution of book categories and
# identify the most frequent labels
books["categories"].value_counts().reset_index()
books["categories"].value_counts().reset_index().query("count > 50")

In [ ]:
# Map the original book categories to two
# high-level classes: Fiction and Nonfiction.
category_mapping = {'Fiction': "Fiction",
                    'Juvenile Fiction': "Children's Fiction",
                    'Biography & Autobiography': "Nonfiction",
                    'History': "Nonfiction",
                    'Literary Criticism': "Nonfiction",
                    'Philosophy': "Nonfiction",
                    'Religion': "Nonfiction",
                    'Comics & Graphic Novels': "Fiction",
                    'Drama': "Fiction",
                    'Juvenile Nonfiction': "Children's Nonfiction",
                    'Science': "Nonfiction",
                    'Poetry': "Fiction"}

books["simple_categories"] = books["categories"].map(category_mapping)

books[~(books["simple_categories"].isna())]

In [ ]:
# Load a pretrained zero-shot classification model.
# The model predicts the most suitable category
# without requiring task-specific training.
from transformers import pipeline

pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

In [ ]:
# Inspect a few example book descriptions that
# belong to the Fiction category.
sequence = books.loc[books["simple_categories"] == "Fiction", "description"].reset_index(drop=True)[:5]
sequence

In [ ]:
fiction_categories = ["Fiction", "Nonfiction"]


In [ ]:
import numpy as np


def generate_predictions(sequence, categories):
    """
    Predict whether a book description belongs to
    Fiction or Nonfiction using zero-shot classification.

    Parameters
    ----------
    sequence : str
        Book description.

    categories : list[str]
        Candidate labels.

    Returns
    -------
    str
        Predicted category.
    """
    predictions = pipe(sequence, categories)
    max_index = np.argmax(predictions["scores"])
    max_label = predictions["labels"][max_index]
    return max_label

In [ ]:
# Predict categories for a sample of Fiction books
# and store the results for evaluation.
from tqdm import tqdm

actual_cats = []
predicted_cats = []
for i in tqdm(range(0, 300)):
    sequence = books.loc[books["simple_categories"] == "Fiction", "description"].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence, fiction_categories)]
    actual_cats += ["Fiction"]

In [ ]:
# Repeat the evaluation using Nonfiction books.
for i in tqdm(range(0, 300)):
    sequence = books.loc[books["simple_categories"] == "Nonfiction", "description"].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence, fiction_categories)]
    actual_cats += ["Nonfiction"]


In [ ]:
# Combine the predicted and true labels into a
# single evaluation DataFrame.
predictions_df = pd.DataFrame({"actual_categories": actual_cats, "predicted_categories": predicted_cats})
predictions_df

In [1]:
# Calculate the classification accuracy of the
# zero-shot model.
predictions_df["correct_prediction"] = (np.where(predictions_df["actual_categories"] == predictions_df["predicted_categories"], 1, 0))

predictions_df["correct_prediction"].sum() / len(predictions_df)

NameError: name 'predictions_df' is not defined

In [ ]:
# Select books that do not yet have a simplified
# category label.
isbns = []
predicted_cats = []

missing_cats = books.loc[books["simple_categories"].isna(), ["isbn13", "description"]].reset_index(drop=True)

In [ ]:
# Predict the missing category for every
# unlabeled book description.
for i in tqdm(range(0, len(missing_cats))):
    sequence = missing_cats["description"][i]
    predicted_cats += [generate_predictions(sequence, fiction_categories)]
    isbns += [missing_cats["isbn13"][i]]

In [ ]:
# Store the predicted categories together with
# their corresponding ISBNs.
missing_predicted_df = pd.DataFrame({"isbn13": isbns, "predicted_categories": predicted_cats})
missing_predicted_df

In [ ]:
# Merge the predicted categories back into the original dataset.
# Existing labels remain unchanged while missing labels are replaced by the predictions.
books = pd.merge(books, missing_predicted_df, on="isbn13", how="left")
books["simple_categories"] = np.where(books["simple_categories"].isna(), books["predicted_categories"],
                                      books["simple_categories"])
books = books.drop(columns=["predicted_categories"])

In [ ]:
books.to_csv("books_with_categories.csv", index=False)